In [1]:
import numpy as np
from tqdm import tqdm

The neural network we are going to recreate is made up of only one neuron. Basically, it is just a perceptron and we are going to try to optimize that.

An Important insight is that these neural networks cannot really generate negative values here. It is a leaky abstraction. Because of the fact that we are using relu activation we are unable to generate a negative value. Because it just clips it down to a negative value. And, also the gradient is such that if the activations are negative. Then, the gradient should be zero.

In [1054]:
X = np.array([[1, 3, 1], [2, 2, 2], [3, 5, 3], [4, 1, 4], [5, 1, 5], [-1, 3, 6], [-2, 10, 7], [1, -4, 8], [2, 5, 9], [0, 3, 10], [1, 0, 11], [-2, 20, 12]])
Y = np.array([11, 13, 19, 18, 21, -2, 0, -3, 9, -2, -1, 5])

In [1055]:
weight = np.array([1, 1, 1])
bias = np.array(1)

In [1056]:
N = len(X)
learning_rate = 0.001

In [1060]:
for i in tqdm(range(100000)):
    activations = np.maximum(0, X @ weight + bias)
    error = activations - Y

    error[activations <= 0] = 0


    gradient_w = 2/N *  np.sum( error * X.T , axis=1)
    gradient_b = 2/N *  np.sum(error)





    weight = weight - learning_rate*gradient_w
    bias = bias - learning_rate*gradient_b

    # if i % 100 == 0:
    # print(f"Weight = {weight} Bias = {bias}")

100%|██████████| 100000/100000 [00:02<00:00, 36883.38it/s]


In [1061]:
print(f"Weight = {weight} Bias = {bias}")

Weight = [ 4.00000012  1.00000004 -1.00000001] Bias = 4.9999995530323265


Another important insight is that if it jumps too far in the negative, it is hard for it to recover because the gradient then just collapses immmediately.

In [21]:
X = np.array([[1, 3, 1], [2, 2, 2], [3, 5, 3], [4, 1, 4], [5, 1, 5], [-1, 3, 6], [-2, 10, 7], [1, -4, 8], [2, 5, 9], [0, 3, 10], [1, 0, 11], [-2, 20, 12]])
Y = np.array([[123], [129], [243], [168], [201], [169], [357], [74], [322], [250], [202], [707]])


In [22]:
# W1 = np.array([[2, 1, 5], [1, 4, 0], [3, 2, 1], [1, 3, 2]])
# B1 = np.array([1, 1, 0, 1])

# W2 = np.array([[1, 2, 3, 4]])
# B2 = np.array([2])
# W1.shape, W2.shape, B1.shape, B2.shape

In [23]:
# for i in range(12):
#     f0 = W1 @ X[i] + B1
#     h1 = np.maximum(f0, 0)
#     print(W2 @ h1 + B2)


In [39]:
X = X.reshape(12, 3, 1)
Y = Y.reshape(12, 1, 1)

In [145]:
class NeuralNetwork():
    def __init__(self, X, Y):

        # np.random.randint(1, 5, size=(4, 3))

        W0 = np.random.uniform(1, 5, size=(4, 3))
        B0 = np.random.uniform(1, 5, size=(4, 1))
        W1 = np.random.uniform(1, 5, size=(1, 4))
        B1 = np.random.uniform(1, 5, size=(1, 1))

        self.W = [W0, W1]
        self.B = [B0, B1]

        self.X = X
        self.Y = Y

        self.f = []
        self.h = []

        self.gradient_f = []
        self.gradient_W = []
        self.gradient_B = []

    def forward_pass(self, x):

        f0 = self.W[0] @ x + self.B[0]
        h1 = np.maximum(f0, 0)
        f1 = self.W[1] @ h1 + self.B[1]

        self.f = [f0, f1]
        self.h = [0, h1]

    def get_gradients(self, x, y):
        diagonal_matrix = np.diag(self.f[0].flatten())
        np.fill_diagonal(diagonal_matrix, np.maximum(diagonal_matrix.diagonal(), 0))

        gradient_f1 = 2 * (self.f[1] - y)
        gradient_f0 = gradient_f1 @ self.W[1] @ diagonal_matrix

        gradient_B0 = gradient_f0.T
        gradient_B1 = gradient_f1.T

        gradient_W0 = (x @ gradient_f0).T
        gradient_W1 = (self.h[1] @ gradient_f1).T

        self.gradient_f.append( [gradient_f0, gradient_f1] )
        self.gradient_W.append( [gradient_W0, gradient_W1] )
        self.gradient_B.append( [gradient_B0, gradient_B1] )

    def gradient_descent(self, learning_rate= 0.00000001):

        gradient_B0 = np.mean([sublist[0] for sublist in self.gradient_B])
        gradient_B1 = np.mean([sublist[1] for sublist in self.gradient_B])
        gradient_W0 = np.mean([sublist[0] for sublist in self.gradient_W])
        gradient_W1 = np.mean([sublist[1] for sublist in self.gradient_W])

        self.B[0] = self.B[0] - learning_rate*gradient_B0
        self.B[1] = self.B[1] - learning_rate*gradient_B1
        self.W[0] = self.W[0] - learning_rate*gradient_W0
        self.W[1] = self.W[1] - learning_rate*gradient_W1

    def train(self, epochs = 100):
        for _ in tqdm(range(epochs)):

            self.gradient_f = []
            self.gradient_W = []
            self.gradient_B = []

            for i in range(len(X)):

                self.forward_pass(X[i])
                self.get_gradients(X[i], Y[i])

            self.gradient_descent()

In [146]:
nn = NeuralNetwork(X, Y)


In [147]:
nn.train(epochs=100000)

  0%|          | 0/100000 [00:00<?, ?it/s]

100%|██████████| 100000/100000 [02:08<00:00, 776.92it/s]


In [149]:
for i in range(12):
    nn.forward_pass(X[i])
    print(nn.h[1])
    print(nn.f[1], Y[i])

[[29.97715818]
 [30.8030549 ]
 [35.1119213 ]
 [27.29460637]]
[[115.8026418]] [[123]]
[[37.63979742]
 [41.10179103]
 [37.74172164]
 [34.98474516]]
[[142.72322715]] [[129]]
[[66.79531605]
 [72.38161682]
 [70.81769155]
 [63.3777933 ]]
[[252.94534181]] [[243]]
[[58.33829574]
 [66.94453571]
 [50.61286471]
 [55.54075008]]
[[217.38978019]] [[168]]
[[71.37415483]
 [82.48854425]
 [60.85420744]
 [68.4066162 ]]
[[265.13574787]] [[201]]
[[45.69647032]
 [53.02354217]
 [54.10983213]
 [36.37116391]]
[[175.77584939]] [[169]]
[[ 82.21344456]
 [ 89.42744176]
 [108.42945652]
 [ 69.68061479]]
[[318.22617027]] [[357]]
[[34.15564955]
 [47.39465236]
 [21.31172084]
 [25.87280481]]
[[125.49010847]] [[74]]
[[ 95.55048728]
 [110.14611263]
 [100.05694511]
 [ 85.32021696]]
[[360.66451709]] [[322]]
[[76.64277097]
 [91.41405258]
 [81.27143041]
 [64.15486852]]
[[290.14973549]] [[250]]
[[73.5589705 ]
 [91.22224388]
 [68.67814596]
 [61.49355263]]
[[275.41955619]] [[202]]
[[165.79637898]
 [179.95766902]
 [212.74530638]


In [150]:
nn.W, nn.B

([array([[7.0657119 , 5.37321985, 5.97014719],
         [7.92850792, 5.24527241, 7.61550062],
         [4.60125755, 7.61154239, 5.64008518],
         [7.8932533 , 5.17572734, 4.97261283]]),
  array([[1.88722587, 0.6378135 , 0.57770152, 0.56846071]])],
 [array([[ 0.82163955],
         [-0.47677089],
         [ 2.03595139],
         [-1.09844177]]),
  array([[3.7822474]])])

In [ ]:
# W1 = np.array([[2, 1, 5], [1, 4, 0], [3, 2, 1], [1, 3, 2]])
# B1 = np.array([1, 1, 0, 1])

# W2 = np.array([[1, 2, 3, 4]])
# B2 = np.array([2])
# W1.shape, W2.shape, B1.shape, B2.shape

In [127]:
nn.gradient_B[0], nn.gradient_W[0]

([array([[  147.25841932],
         [-1738.69861329],
         [  508.52600507],
         [ -913.80358899]]),
  array([[-18.172784]])],
 [array([[  147.25841932,   441.77525795,   147.25841932],
         [-1738.69861329, -5216.09583988, -1738.69861329],
         [  508.52600507,  1525.57801521,   508.52600507],
         [ -913.80358899, -2741.41076697,  -913.80358899]]),
  array([[-1207.97708716, -1284.36098862, -1177.44804305, -1176.50745259]])])